In [1]:
import pandas as pd


In [2]:
import numpy as np


In [3]:
from scipy.sparse import csr_matrix

In [4]:
train = pd.read_csv("train.csv")

In [5]:
train.head()

,msno,song_id,source_system_tab,source_screen_name,source_type,target
0,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,BBzumQNXUHKdEBOB7mAJuzok+IJA1c2Ryg/yzTF6tik=,explore,Explore,online-playlist,1
1,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,bhp/MpSNoqoxOIB+/l8WPqu6jldth4DIpCm3ayXnJqM=,my library,Local playlist more,local-playlist,1
2,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,JNWfrrC7zNN7BdMpsISKa4Mw+xVJYNnxXh3/Epw7QgY=,my library,Local playlist more,local-playlist,1
3,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,2A87tzfnJTSWqD7gIZHisolhe4DMdzkbd6LzO1KHjNs=,my library,Local playlist more,local-playlist,1
4,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,3qm6XTZ6MOCU11x8FIVbAGH5l5uMkT3/ZalWG1oo2Gc=,explore,Explore,online-playlist,1


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7377418 entries, 0 to 7377417
Data columns (total 6 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   msno                object
 1   song_id             object
 2   source_system_tab   object
 3   source_screen_name  object
 4   source_type         object
 5   target              int64 
dtypes: int64(1), object(5)
memory usage: 337.7+ MB


In [7]:
train.describe()

,target
count,7.377418e+06
mean,5.035171e-01
std,4.999877e-01
min,0.000000e+00
25%,0.000000e+00
50%,1.000000e+00
75%,1.000000e+00
max,1.000000e+00


In [8]:
train['msno'].nunique()

30755

In [9]:
train['song_id'].nunique()

359966

In [10]:
df = train[["msno", "song_id", "target"]].copy()

In [11]:
df.isnull().sum()

msno       0
song_id    0
target     0
dtype: int64

In [12]:
df = df[df["target"] == 1]

In [13]:
df = df.drop_duplicates(["msno", "song_id"])

In [14]:
df.head()

,msno,song_id,target
0,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,BBzumQNXUHKdEBOB7mAJuzok+IJA1c2Ryg/yzTF6tik=,1
1,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,bhp/MpSNoqoxOIB+/l8WPqu6jldth4DIpCm3ayXnJqM=,1
2,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,JNWfrrC7zNN7BdMpsISKa4Mw+xVJYNnxXh3/Epw7QgY=,1
3,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,2A87tzfnJTSWqD7gIZHisolhe4DMdzkbd6LzO1KHjNs=,1
4,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,3qm6XTZ6MOCU11x8FIVbAGH5l5uMkT3/ZalWG1oo2Gc=,1


In [15]:
df.shape

(3714656, 3)

In [16]:
df['msno'] = df['msno'].astype('category')

In [17]:
df['song_id'] = df['song_id'].astype('category')

In [18]:
df['user_id'] = df['msno'].cat.codes
df['item_id'] = df['song_id'].cat.codes

In [19]:
user_map = dict(enumerate(df['msno'].cat.categories))
song_map = dict(enumerate(df['song_id'].cat.categories))

In [20]:
df[['msno', 'user_id']].head()
df[['song_id', 'item_id']].head()

,song_id,item_id
0,BBzumQNXUHKdEBOB7mAJuzok+IJA1c2Ryg/yzTF6tik=,46606
1,bhp/MpSNoqoxOIB+/l8WPqu6jldth4DIpCm3ayXnJqM=,139055
2,JNWfrrC7zNN7BdMpsISKa4Mw+xVJYNnxXh3/Epw7QgY=,75249
3,2A87tzfnJTSWqD7gIZHisolhe4DMdzkbd6LzO1KHjNs=,14771
4,3qm6XTZ6MOCU11x8FIVbAGH5l5uMkT3/ZalWG1oo2Gc=,20750


In [21]:
df.groupby("song_id")["item_id"].nunique().max()

C:\Users\BUSE\AppData\Local\Temp\ipykernel_10416\1410588550.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("song_id")["item_id"].nunique().max()


1

In [22]:
n_users = df['user_id'].nunique()
n_items = df['item_id'].nunique()

In [23]:
interaction_matrix = csr_matrix(
    (df['target'].astype(np.float32), (df['user_id'], df['item_id'])),
    shape=(n_users, n_items)
)

In [24]:
print("User-Item matrix:", interaction_matrix.shape)

User-Item matrix: (27113, 223723)


In [25]:
item_user_matrix = interaction_matrix.T.tocsr()

In [26]:
print("Item-User matrix:", item_user_matrix.shape)

Item-User matrix: (223723, 27113)
